### 1. The Mini-Batch Derivative Formulas
Instead of looking at one sample $i$, we compute the average gradient over a mini-batch $B$ containing $m$ samples.

* **Intercept Derivative:**
$$\frac{\partial J_B}{\partial b} = \frac{1}{m} \sum_{i \in B} -2(y_i - \hat{y}_i) = \frac{2}{m} \sum_{i \in B} (\hat{y}_i - y_i)$$

* **Coefficients Derivative Vector:**
$$\frac{\partial J_B}{\partial w} = \frac{1}{m} \sum_{i \in B} -2(y_i - \hat{y}_i) \cdot X_i = \frac{2}{m} \sum_{i \in B} (\hat{y}_i - y_i) \cdot X_i$$

---

### 2. The Mini-Batch Parameter Update Formulas
The parameters are updated once per mini-batch by subtracting the averaged gradient multiplied by the learning rate ($\eta$). The double negatives cancel out when using the error term $(y_i - \hat{y}_i)$.

* **Intercept Update:**
$$b_{\text{new}} = b_{\text{old}} + \frac{2\eta}{m} \sum_{i \in B} (y_i - \hat{y}_i)$$

* **Coefficients Update:**
$$w_{\text{new}} = w_{\text{old}} + \frac{2\eta}{m} \sum_{i \in B} (y_i - \hat{y}_i) \cdot X_i$$

---

### 3. Vectorized Matrix Notation
In actual code implementation, loops are avoided by using matrix operations for the mini-batch matrix $X_B$ (size $m \times d$) and target vector $y_B$ (size $m \times 1$).

* **Predictions Vector:**
$$\hat{y}_B = X_B w + b$$

* **Intercept Update (Vectorized):**
$$b_{\text{new}} = b_{\text{old}} + \frac{2\eta}{m} \sum (\text{errors})$$

* **Coefficients Update (Vectorized):**
$$w_{\text{new}} = w_{\text{old}} + \frac{2\eta}{m} X_B^T (y_B - \hat{y}_B)$$

---

### 4. Key Implementation Details
* **Batch Size ($m$):** Typically chosen in powers of 2 (e.g., 32, 64, 128) to optimize hardware memory performance.
* **Shuffling:** The dataset must be randomly shuffled at the start of each epoch before splitting into mini-batches to prevent biased updates.
* **Compromise:** This method offers a middle ground—smoother convergence than Stochastic GD (single sample) and much faster computation than Full-Batch GD.


In [26]:
from sklearn.datasets import load_diabetes

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import random

In [27]:
X, y = load_diabetes(return_X_y = True)

In [28]:
print(X.shape)
print(y.shape)

(442, 10)
(442,)


In [29]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [30]:
reg = LinearRegression()
reg.fit(X_train, y_train)

LinearRegression()

In [31]:
print(reg.coef_) # Beta_1ton
print(reg.intercept_) # Beta_0

[  -9.15865318 -205.45432163  516.69374454  340.61999905 -895.5520019
  561.22067904  153.89310954  126.73139688  861.12700152   52.42112238]
151.88331005254167


In [32]:
y_pred = reg.predict(X_test)
r2_score(y_test, y_pred)

0.4399338661568968

## Creating own regression class

In [39]:
class MiniBatchGradientDescent:

  def __init__(self, batch_size, learning_rate, epochs):
    self.coef_ = None
    self.intercept_ = None
    self.lr = learning_rate
    self.epochs = epochs
    self.batch_size = batch_size

  def fit(self, X_train, y_train):
    # initialize your coefficients
    self.intercept_ = 0
    self.coef_ = np.ones(X_train.shape[1])

    # training loop
    for i in range (self.epochs):

      # weights will be updated times as many number of batches
      for j in range (X_train.shape[0]//self.batch_size): # number of batches

        idx = random.sample(range(1, X_train.shape[0]), self.batch_size)

        y_hat = np.dot(X_train[idx], self.coef_) + self.intercept_

        intercept_derivative = -2 * np.mean((y_train[idx] - y_hat))
        self.intercept_ = self.intercept_ - (self.lr * intercept_derivative)

        coef_derivative = -2 * np.dot((y_train[idx] - y_hat), X_train[idx])
        self.coef_ = self.coef_ - (self.lr * coef_derivative)

    print(self.intercept_, self.coef_)

  def predict(self, X_test):
    return np.dot(X_test, self.coef_) + self.intercept_


In [60]:
mbgd = MiniBatchGradientDescent(batch_size = (X_train.shape[0]//20), epochs = 100, learning_rate = 0.01)

In [61]:
mbgd.fit(X_train, y_train)

150.93532776685228 [  22.85869169 -140.05165316  453.05262236  306.30717236  -26.66829001
  -90.04286327 -191.76628551  115.3430692   399.80929468  122.03724716]


In [62]:
y_pred = mbgd.predict(X_test)

In [63]:
r2_score(y_test, y_pred)

0.45421779947624674

## Mini-Batch Gradient Descent Using Sklearn

In [ ]:
from sklearn.linear_model import SGDRegressor

In [ ]:
reg = SGDRegressor(learning_rate = 'constant', eta0 = 0.2)

In [ ]:
# Use .partial_fit to train a subset of data in each iteration
batch_size = 35

for i in range(100):

    idx = random.sample(range(X_train.shape[0]),batch_size)
    reg.partial_fit(X_train[idx],y_train[idx])

In [64]:
reg.intercept_

np.float64(151.88331005254167)

In [65]:
reg.coef_

array([  -9.15865318, -205.45432163,  516.69374454,  340.61999905,
       -895.5520019 ,  561.22067904,  153.89310954,  126.73139688,
        861.12700152,   52.42112238])

In [66]:
y_pred = reg.predict(X_test)

In [67]:
r2_score(y_test, y_pred)

0.4399338661568968